## EXP 5: Char-level text generation (SimpleRNN)

In [ ]:
"""
Aim: To build a character-level language model using a SimpleRNN in TensorFlow. The model learns from the Shakespeare text 
and generates new text from a starting phrase.

Algorithm:

1. Import TensorFlow and NumPy, then load the Shakespeare text file.
2. Find all unique characters and give each one a number (vocabulary).
3. Convert the whole text into numbers using this mapping.
4. Cut the text into sequences of 100 characters. The target is the same sequence shifted by one character.
5. Shuffle the sequences and group them into batches of 64.
6. Build the model with an Embedding layer (128), a SimpleRNN layer (256 units) and a Dense output layer (65 units).
7. Compile with the Adam optimizer and sparse categorical cross-entropy loss (with logits), then train for 10 epochs.
8. Start with "Shall ", predict one character at a time using the last 100 characters, and add each prediction to the text.

Result:

Thus, a character-level language model using SimpleRNN was successfully developed and trained on the given text corpus. The trained model
successfully generated new text based on the learned character patterns.
"""

import numpy as np, tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.losses import SparseCategoricalCrossentropy
tf.keras.utils.set_random_seed(42)

text = open(r"shakespeare.txt", encoding="utf-8").read()
vocab = sorted(set(text)); V = len(vocab)
c2i = {c: i for i, c in enumerate(vocab)}
ids = np.array([c2i[c] for c in text], dtype="int32")
print("Corpus length:", len(text), "| Vocab size:", V)

w = np.lib.stride_tricks.sliding_window_view(ids, 101)
ds = tf.data.Dataset.from_tensor_slices((w[:, :-1], w[:, 1:])).shuffle(10000).batch(64, drop_remainder=True)

model = Sequential([
    Embedding(V, 128),
    SimpleRNN(256, return_sequences=True),
    Dense(V)])
model.compile(optimizer="adam", loss=SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])
model.summary()
model.fit(ds, epochs=25)

def generate(start, n=200):
    out = start
    for _ in range(n):
        x = np.array([[c2i[c] for c in out[-100:]]])
        out += vocab[model.predict(x, verbose=0)[0, -1].argmax()]
    return out

print("\nGenerated Text:\n", generate("Shall ", 200))

## EXP 6: IMDB sentiment (LSTM)

In [ ]:
"""
Aim: To build an LSTM model that classifies IMDb movie reviews as Positive or Negative, and to test it on a custom sentence.

Algorithm:

1. Import TensorFlow and load the IMDb dataset, keeping only the top 10,000 most common words.
2. Pad every review to a fixed length of 200 words.
3. Build the model with an Embedding layer (10,000 × 128), an LSTM layer (64 units) and a Dense layer (1 unit, sigmoid).
4. Compile with the Adam optimizer, binary cross-entropy loss and accuracy as the metric.
5. Train for 3 epochs with batch size 64.
6. Evaluate on the test data and print the test accuracy.
7. Load the IMDb word index and convert the sentence "This movie was super" into numbers, then pad it to 200.
8. Predict the score. If the score is 0.5 or more, the sentiment is Positive, otherwise Negative.

Result:

Thus, an LSTM-based sentiment analysis model was successfully developed and trained using the IMDb movie review dataset. The trained model 
successfully classified movie reviews into positive or negative sentiments.
"""

import numpy as np
from tensorflow.keras.datasets import imdb
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences
tf.keras.utils.set_random_seed(42)

(xtr, ytr), (xte, yte) = imdb.load_data(num_words=10000)
xtr, xte = pad_sequences(xtr, maxlen=200), pad_sequences(xte, maxlen=200)

model = Sequential([
    Embedding(10000, 128),
    LSTM(64),
    Dense(1, activation="sigmoid")])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(xtr, ytr, epochs=3, batch_size=64)
print("Test Accuracy:", model.evaluate(xte, yte)[1])

wi = imdb.get_word_index()
def encode(sentence):
    ids = [1]
    for w in sentence.lower().split():
        i = wi.get(w.strip(".,!?"), None)
        ids.append(i + 3 if i is not None and i + 3 < 10000 else 2)
    return pad_sequences([ids], maxlen=200)

sentence = "This movie was super"
p = model.predict(encode(sentence))[0][0]
print("Sentence:", sentence, "\nSentiment:", "Positive" if p >= 0.5 else "Negative", "\nScore:", p)

## EXP 7: POS tagging (Encoder-Decoder LSTM)

In [ ]:
"""
Aim: To build an Encoder–Decoder LSTM model that gives each word in a sentence its POS tag, using the Penn Treebank corpus.

Algorithm:

1. Load the tagged sentences from the Penn Treebank and split each into a list of words and a list of tags.
2. Split the data into 80% training and 20% testing.
3. Create a word tokenizer (using the training data only) and a tag tokenizer, and convert both to numbers.
4. Pad all sequences to the longest training sentence length.
5. Build the Encoder with an Embedding layer (128) and an LSTM layer (128 units) that returns its outputs and states.
6. Build the Decoder with an LSTM layer (128 units) that starts from the encoder states, followed by a Dense softmax layer that predicts
   a tag for each word.
7. Compile with the Adam optimizer and sparse categorical cross-entropy loss, then train for 5 epochs with batch size 32 and 10% validation data.
8. Evaluate on the test data, take the highest-probability tag for each word, and compare predicted and actual tags.

Result:

Thus, an Encoder–Decoder LSTM model for Part-of-Speech tagging was successfully developed and trained using the Penn Treebank corpus. 
The trained model successfully predicted the appropriate Part-of-Speech tags for the given sentences.
"""

import nltk, numpy as np
from nltk.corpus import treebank
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
nltk.download("treebank")

sents = treebank.tagged_sents()
X = [[w for w, t in s] for s in sents]
Y = [[t for w, t in s] for s in sents]
Xtr, Xte, Ytr, Yte = train_test_split(X, Y, test_size=0.2, random_state=42)

wtok = Tokenizer(lower=True, oov_token="<UNK>"); wtok.fit_on_texts(Xtr)     
ttok = Tokenizer(lower=False, filters="");       ttok.fit_on_texts(Ytr)
wv, tv = len(wtok.word_index) + 1, len(ttok.word_index) + 1
max_len = max(len(s) for s in Xtr)

def pad(tok, data):
    return pad_sequences(tok.texts_to_sequences(data), maxlen=max_len, padding="post", truncating="post")
Xtr_p, Xte_p, Ytr_p, Yte_p = pad(wtok, Xtr), pad(wtok, Xte), pad(ttok, Ytr), pad(ttok, Yte)
print("Vocab:", wv, "| Tags:", tv, "| Max len:", max_len, "| Input:", Xtr_p.shape)

inp = Input((max_len,))
enc_out, h, c = LSTM(128, return_sequences=True, return_state=True)(Embedding(wv, 128, mask_zero=True)(inp))
dec_out = LSTM(128, return_sequences=True)(enc_out, initial_state=[h, c])
model = Model(inp, Dense(tv, activation="softmax")(dec_out))
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()
model.fit(Xtr_p, Ytr_p[..., None], validation_split=0.1, epochs=5, batch_size=32)
loss, acc = model.evaluate(Xte_p, Yte_p[..., None])
print("Test Loss:", loss, "| Test Accuracy:", acc)

id2tag = {v: k for k, v in ttok.word_index.items()}
pred = model.predict(Xte_p).argmax(-1)
i = 782                                                         
print("\nWords:", Xte[i], "\nActual:", Yte[i], "\nPredicted:", [id2tag.get(t, "UNK") for t in pred[i][:len(Xte[i])]])

## EXP 8: English -> Tamil translation (Seq2Seq LSTM)

In [ ]:
"""
Aim: To build an Encoder–Decoder LSTM model that translates English sentences into Tamil after training on paired sentences.

Algorithm:

1. Read the English–Tamil sentence pairs from the tab-separated file, and remove empty lines and duplicate pairs.
2. Add <start> and <end> tokens to every Tamil sentence.
3. Create separate tokenizers for English and Tamil, convert the sentences to numbers, and pad them to the maximum length.
4. Make the decoder input from the Tamil sequence without its last token, and the decoder output from the sequence 
   without its first token (shifted by one).
5. Build the Encoder with an Embedding layer (64) and an LSTM layer (128 units), and pass its hidden and cell states to the Decoder.
6. Build the Decoder with an Embedding layer (64), an LSTM layer (128 units) and a Dense softmax layer.
7. Compile with the Adam optimizer and sparse categorical cross-entropy loss, then train for 100 epochs with batch size 32.
8. To translate, start with <start>, predict one Tamil word at a time, and stop when <end> is generated.

Result:

Thus, an LSTM-based English-to-Tamil neural machine translation model was successfully developed and trained using paired sentence 
data. The trained model successfully generated Tamil translations for the given English test sentences.
"""

import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

pairs = []
for line in open(r"eng_tamil_pairs.txt", encoding="utf-8"):     # <- set your path
    p = line.strip().split("\t", 1)
    if len(p) == 2: pairs.append((p[0].strip(), p[1].strip()))
print("Pairs:", len(pairs))
pairs = list(dict.fromkeys(pairs))                              
print("Unique pairs:", len(pairs))
english = [e for e, t in pairs]
tamil = [f"<start> {t} <end>" for e, t in pairs]

def tokenize(texts):
    tok = Tokenizer(filters=""); tok.fit_on_texts(texts)
    seq = tok.texts_to_sequences(texts); L = max(len(s) for s in seq)
    return tok, pad_sequences(seq, maxlen=L, padding="post"), L
et, eng, max_eng = tokenize(english)
tt, tam, max_tam = tokenize(tamil)
ev, tv = len(et.word_index) + 1, len(tt.word_index) + 1
print("Max len:", max_eng, max_tam, "| Vocab:", ev, tv)

dec_in_data, dec_out_data = tam[:, :-1], tam[:, 1:]             

# Encoder
enc_in = Input((max_eng,))
_, h, c = LSTM(128, return_state=True)(Embedding(ev, 64, mask_zero=True)(enc_in))   
# Decoder
dec_in = Input((max_tam - 1,))
dec = LSTM(128, return_sequences=True)(Embedding(tv, 64, mask_zero=True)(dec_in), initial_state=[h, c])  
model = Model([enc_in, dec_in], Dense(tv, activation="softmax")(dec))
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()
model.fit([eng, dec_in_data], dec_out_data[..., None], epochs=50, batch_size=32, verbose=1)

rev = {i: w for w, i in tt.word_index.items()}
start, end = tt.word_index["<start>"], tt.word_index["<end>"]
def translate(sentence):
    src = pad_sequences(et.texts_to_sequences([sentence]), maxlen=max_eng, padding="post")
    tgt = [start]
    for _ in range(max_tam - 1):                                
        t = pad_sequences([tgt], maxlen=max_tam - 1, padding="post")
        nxt = model.predict([src, t], verbose=0)[0, len(tgt) - 1].argmax()
        if nxt in (0, end): break
        tgt.append(nxt)
    return " ".join(rev[i] for i in tgt[1:])

for s in ["Get down.", "Goodbye and take care.", "Let's meet again soon.", "Good morning.", "Thank you.", "How are you?", "hospital"]:
    print("\nEnglish :", s, "\nTamil   :", translate(s))

## EXP 9: GAN data augmentation (Fashion-MNIST)

In [ ]:
"""
Aim: To build a GAN that generates synthetic Fashion-MNIST images, and to check whether adding these images to the training data improves classifier accuracy.

Algorithm:

1. Load the Fashion-MNIST dataset. Scale the images to [-1, 1] for the GAN and to [0, 1] for the classifier.
2. Build the Generator, which turns a 100-value noise vector into a 28×28 image.
3. Build the Discriminator, which uses Conv2D, LeakyReLU, Dropout and a sigmoid output to decide if an image is real or fake.
4. Join the Generator and Discriminator into a GAN model, keeping the Discriminator frozen inside it.
5. Train for 50 epochs with batch size 128. In each step, train the Discriminator on real images (label 0.9) and fake images (label 0), 
   then train the Generator to make the Discriminator say "real".
6. Display 16 generated images, then generate 5,000 synthetic images.
7. Train a CNN classifier on the original data only (3 epochs), then train an identical CNN on the original data plus the 5,000 synthetic 
   images (labelled using the first classifier).
8. Test both classifiers on the test set, then compare their accuracies and show them in a bar chart.

Result:

Thus, a GAN-based synthetic data generation and data augmentation model was successfully developed and trained using the Fashion-MNIST 
dataset. The trained GAN successfully generated synthetic images, which were used to augment the training data for image classification.
"""

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input, Dense, BatchNormalization, LeakyReLU, Reshape,
    Conv2DTranspose, Conv2D, Dropout, Flatten, MaxPooling2D
)

tf.keras.utils.set_random_seed(42)

(xr, ytr), (xt, yte) = tf.keras.datasets.fashion_mnist.load_data()
xr, xt = xr[..., None].astype("float32"), xt[..., None].astype("float32")
x_gan = (xr - 127.5) / 127.5                                    
x_cls, x_test = xr / 255.0, xt / 255.0                          

G = Sequential([layers.Input((100,)),
    Dense(7 * 7 * 256, use_bias=False), BatchNormalization(), LeakyReLU(), Reshape((7, 7, 256)),
    Conv2DTranspose(128, 4, 2, padding="same", use_bias=False), BatchNormalization(), LeakyReLU(),
    Conv2DTranspose(64, 4, 2, padding="same", use_bias=False), BatchNormalization(), LeakyReLU(),
    Conv2D(1, 3, padding="same", activation="tanh")])

D = Sequential([layers.Input((28, 28, 1)),            
    Conv2D(64, 4, 2, padding="same"), LeakyReLU(0.2), Dropout(0.3),
    Conv2D(128, 4, 2, padding="same"), LeakyReLU(0.2), Dropout(0.3),
    Flatten(), Dense(1, activation="sigmoid")])
D.compile(optimizer=tf.keras.optimizers.Adam(0.0002, 0.5), loss="binary_crossentropy")

D.trainable = False                                             
GAN = Sequential([G, D])
GAN.compile(optimizer=tf.keras.optimizers.Adam(0.0002, 0.5), loss="binary_crossentropy")

noise = lambda n: np.random.normal(0, 1, (n, 100)).astype("float32")
for epoch in range(50):
    for i in range(0, len(x_gan), 128):
        real = x_gan[i:i + 128]; n = len(real)
        fake = G(noise(n), training=True).numpy()               
        D.trainable = True
        D.train_on_batch(real, np.full((n, 1), 0.9))           
        D.train_on_batch(fake, np.zeros((n, 1)))
        D.trainable = False
        GAN.train_on_batch(noise(n), np.ones((n, 1)))           
    print("Epoch:", epoch + 1)

gen = (G.predict(noise(16), verbose=0) + 1) / 2                 
plt.figure(figsize=(8, 8))
for i in range(16):
    plt.subplot(4, 4, i + 1); plt.imshow(gen[i].squeeze(), cmap="gray"); plt.axis("off")
plt.suptitle("GAN Generated Images"); plt.tight_layout(); plt.show()

num_fake = 5000
fake_images = (G.predict(noise(num_fake), verbose=0) + 1) / 2
print("Generated images:", fake_images.shape)

def classifier():
    m = Sequential([layers.Input((28, 28, 1)),
        Conv2D(32, 3, activation="relu"), MaxPooling2D(),
        Conv2D(64, 3, activation="relu"), MaxPooling2D(),
        Flatten(), layers.Dense(64, activation="relu"), Dense(10, activation="softmax")])
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

print("Training original classifier...")
m_orig = classifier(); m_orig.fit(x_cls, ytr, epochs=3, batch_size=128)
acc_orig = m_orig.evaluate(x_test, yte, verbose=0)[1]


fake_labels = m_orig.predict(fake_images, verbose=0).argmax(1)
# fake_labels = np.random.choice(ytr, num_fake)

x_aug = np.concatenate([x_cls, fake_images]); y_aug = np.concatenate([ytr, fake_labels])
idx = np.random.permutation(len(x_aug)); x_aug, y_aug = x_aug[idx], y_aug[idx]
print("Augmented data:", x_aug.shape)

print("Training augmented classifier...")
m_aug = classifier(); m_aug.fit(x_aug, y_aug, epochs=3, batch_size=128)
acc_aug = m_aug.evaluate(x_test, yte, verbose=0)[1]

print(f"\nOriginal Accuracy: {acc_orig*100:.2f} %\nGAN Augmented Accuracy: {acc_aug*100:.2f} %\nChange: {(acc_aug-acc_orig)*100:.2f} %")
plt.bar(["Original", "GAN Augmented"], [acc_orig * 100, acc_aug * 100])
plt.ylabel("Accuracy (%)"); plt.title("Classification Accuracy"); plt.ylim(0, 100); plt.show()